# Music Annotation Project V9.0A
## YOLO Mask Authority and Transfer Audit

**Runtime:** GPU recommended for YOLO inference; CPU is sufficient if frozen detector outputs already exist.

V9.0A is a **development / shadow-mode mask-authority transfer experiment**. It keeps the qualified V8.8 correction engine frozen and changes one causal variable: the intervention mask.

Paired mask arms:
1. reviewed authoritative mask (comparator),
2. raw frozen-YOLO proposal,
3. confidence-gated frozen-YOLO proposal,
4. reviewer-corrected YOLO mask when available.

**Hard boundaries**
- V8.8 is not retrained, relaxed or retuned.
- Raw YOLO masks are proposals, not authority.
- Detector confidence and correction safety remain separate axes.
- Every arm uses the same annotated page and frozen V8.8 engine.
- Outputs are locked before clean-truth scoring or reviewed-mask comparison metrics are calculated.
- Any pixel outside the mask for an arm remains byte-for-byte identical to the observed page.
- This notebook does not authorise automatic production use.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib scipy


## 1. Configuration and change controls


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, sys, platform, time, math
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
OUTPUT_ROOT = ROOT/'v9_0a_yolo_mask_authority_transfer_audit'

# Set these only to exact frozen run locations when automatic discovery is ambiguous.
V88_RUN_OVERRIDE = None
YOLO_BUNDLE_OVERRIDE = None
COHORT_MANIFEST_OVERRIDE = None

# Expected paired input locations. Adjust only if the project uses different durable paths.
REVIEWED_MASK_DIR = ROOT/'masks'/'corrected'
ANNOTATED_DIR = ROOT/'aligned'
CLEAN_DIR = ROOT/'source'/'clean'  # scoring only, loaded after prediction lock
YOLO_MASK_DIR = ROOT/'human_in_the_middle'/'draft_masks'
YOLO_CONFIDENCE_DIR = ROOT/'human_in_the_middle'/'confidence_maps'
YOLO_CORRECTED_MASK_DIR = ROOT/'production_pilot'/'reviewed_masks'

# Frozen proposal-only confidence gates. These are sensitivity arms, not authority claims.
CONFIDENCE_GATES = [0.90, 0.95, 0.98, 0.995]
PRIMARY_CONFIDENCE_GATE = 0.98
MASK_DILATION = 2
BOOTSTRAP_RESAMPLES = 10000
SEED = 42

# Controls are deliberately fail-closed.
ALLOW_MISSING_CORRECTED_YOLO = True
ALLOW_MISSING_CLEAN_FOR_SHADOW_ONLY = False
PAGE_ALLOWLIST = None

RUN = datetime.now().strftime('%Y%m%d_%H%M%S')+'_v9_0a_mask_authority_transfer'
OUT = OUTPUT_ROOT/RUN
INPUTS=OUT/'inputs'; MASKS=OUT/'masks'; OUTPUTS=OUT/'outputs'; TABLES=OUT/'tables'; AUDIT=OUT/'audit'; VISUAL=OUT/'visual_audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,MASKS,OUTPUTS,TABLES,AUDIT,VISUAL,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
print('Output:',OUT)


## 2. Identity helpers and frozen-artifact discovery

V9.0A refuses to guess a qualified V8.8 implementation. A complete V8.8 run must expose an operational builder entry point or durable output contract plus a release/experiment identity. The YOLO side must expose a frozen detector identity and proposal masks/confidence maps.


In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()

def object_hash(x): return hashlib.sha256(json.dumps(x,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()

def image_map(folder):
    p=Path(folder)
    return {x.stem:x for x in p.iterdir() if x.is_file() and x.suffix.lower() in SUPPORTED} if p.exists() else {}

def latest_complete(root, alternatives):
    hits=[]
    root=Path(root)
    if root.exists():
        for p in root.rglob('*'):
            if not p.is_dir(): continue
            for required in alternatives:
                if all((p/r).exists() for r in required): hits.append(p); break
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

# Flexible but explicit discovery: accepted V8.8 identities must include an identity/manifest plus an operational module.
V88_ALTS=[
 ['release_manifest.json','operational_builder.py'],
 ['experiment_identity.json','operational_builder.py'],
 ['registered_candidate_manifest.json','operational_builder.py'],
 ['release_manifest.json','v88_operational.py'],
]
V88_ROOT=ROOT/'v8_8'
V88=Path(V88_RUN_OVERRIDE) if V88_RUN_OVERRIDE else latest_complete(V88_ROOT,V88_ALTS)
if V88 is None:
    raise RuntimeError('No complete frozen V8.8 operational run found. Set V88_RUN_OVERRIDE to the exact qualified run containing a manifest/identity and operational_builder.py or v88_operational.py.')

YOLO_ROOT=ROOT/'yolo_frozen'
if YOLO_BUNDLE_OVERRIDE:
    YOLO_BUNDLE=Path(YOLO_BUNDLE_OVERRIDE)
else:
    YOLO_BUNDLE=latest_complete(YOLO_ROOT,[['detector_identity.json'],['model_manifest.json'],['inference_manifest.json']])
if YOLO_BUNDLE is None:
    raise RuntimeError('No frozen YOLO detector identity found. Set YOLO_BUNDLE_OVERRIDE to the directory containing detector_identity.json/model_manifest.json/inference_manifest.json.')

v88_identity_file=next((V88/x for x in ['release_manifest.json','experiment_identity.json','registered_candidate_manifest.json'] if (V88/x).exists()),None)
yolo_identity_file=next((YOLO_BUNDLE/x for x in ['detector_identity.json','model_manifest.json','inference_manifest.json'] if (YOLO_BUNDLE/x).exists()),None)
if v88_identity_file is None or yolo_identity_file is None: raise RuntimeError('Frozen identity files unresolved.')

print('V8.8:',V88)
print('YOLO:',YOLO_BUNDLE)
print('V8.8 identity:',v88_identity_file.name,sha256_file(v88_identity_file))
print('YOLO identity:',yolo_identity_file.name,sha256_file(yolo_identity_file))


## 3. Load the frozen V8.8 operational entry point

The operational module must provide `run_v88(observed: np.ndarray, authoritative_mask: np.ndarray, work_dir: Path) -> dict` and return at minimum `corrected`, `m3`, `m5_remove`, and `audit`. This makes V9.0A consume the qualified engine rather than reimplement it.


In [ ]:
import importlib.util
module_path=V88/'operational_builder.py'
if not module_path.exists(): module_path=V88/'v88_operational.py'
spec=importlib.util.spec_from_file_location('v88_frozen_operational',module_path)
v88mod=importlib.util.module_from_spec(spec); spec.loader.exec_module(v88mod)
if not hasattr(v88mod,'run_v88'): raise RuntimeError(f'{module_path} must expose run_v88(observed, authoritative_mask, work_dir).')
V88_MODULE_SHA256=sha256_file(module_path)
print('Frozen V8.8 operational module:',module_path,'SHA-256:',V88_MODULE_SHA256)


## 4. Paired cohort roster and input lock


In [ ]:
observed_map=image_map(ANNOTATED_DIR); reviewed_map=image_map(REVIEWED_MASK_DIR); raw_map=image_map(YOLO_MASK_DIR); corrected_yolo_map=image_map(YOLO_CORRECTED_MASK_DIR)
conf_map={p.stem:p for p in Path(YOLO_CONFIDENCE_DIR).glob('*.npy')} if Path(YOLO_CONFIDENCE_DIR).exists() else {}
clean_map=image_map(CLEAN_DIR)
pages=sorted(set(observed_map)&set(reviewed_map)&set(raw_map)&set(conf_map))
if PAGE_ALLOWLIST is not None: pages=[p for p in pages if p in set(PAGE_ALLOWLIST)]
if not pages: raise RuntimeError('No paired pages with observed + reviewed mask + raw YOLO mask + YOLO confidence map.')
if not ALLOW_MISSING_CLEAN_FOR_SHADOW_ONLY and any(p not in clean_map for p in pages): raise RuntimeError('Clean scoring truth missing for one or more paired pages.')
rows=[]
for page in pages:
    obs=cv2.imread(str(observed_map[page]),0); rev=cv2.imread(str(reviewed_map[page]),0); raw=cv2.imread(str(raw_map[page]),0); conf=np.load(conf_map[page])
    if obs is None or rev is None or raw is None or obs.shape!=rev.shape or obs.shape!=raw.shape or conf.shape!=obs.shape: raise RuntimeError(f'{page}: unreadable input or shape mismatch.')
    rows.append({'page':page,'observed':str(observed_map[page]),'reviewed_mask':str(reviewed_map[page]),'raw_yolo_mask':str(raw_map[page]),'confidence_map':str(conf_map[page]),'reviewer_corrected_yolo':str(corrected_yolo_map[page]) if page in corrected_yolo_map else None,'clean':str(clean_map[page]) if page in clean_map else None,'observed_sha256':sha256_file(observed_map[page]),'reviewed_mask_sha256':sha256_file(reviewed_map[page]),'raw_yolo_mask_sha256':sha256_file(raw_map[page]),'confidence_sha256':sha256_file(conf_map[page])})
cohort=pd.DataFrame(rows)
cohort.to_csv(INPUTS/'paired_cohort_manifest.csv',index=False)
input_lock={'created_utc':datetime.now(timezone.utc).isoformat(),'pages':len(cohort),'v88_identity_sha256':sha256_file(v88_identity_file),'v88_module_sha256':V88_MODULE_SHA256,'yolo_identity_sha256':sha256_file(yolo_identity_file),'cohort_sha256':sha256_file(INPUTS/'paired_cohort_manifest.csv'),'confidence_gates':CONFIDENCE_GATES,'primary_confidence_gate':PRIMARY_CONFIDENCE_GATE,'mask_dilation':MASK_DILATION}
(INPUTS/'input_lock.json').write_text(json.dumps(input_lock,indent=2))
print('Paired pages:',len(cohort)); display(cohort.head())


## 5. Construct paired mask arms

Confidence gating never expands the raw YOLO proposal. It can only subtract low-confidence proposal pixels. Reviewer-corrected YOLO masks remain a separate arm when available.


In [ ]:
def binary(a): return np.where(a>0,255,0).astype(np.uint8)
def confidence_gated(raw,conf,t): return np.where((raw>0)&(conf>=t),255,0).astype(np.uint8)
def mask_metrics(reference,candidate):
    r=reference>0; c=candidate>0; inter=int((r&c).sum()); union=int((r|c).sum()); over=int((c&~r).sum()); miss=int((r&~c).sum())
    return {'reviewed_pixels':int(r.sum()),'candidate_pixels':int(c.sum()),'intersection':inter,'union':union,'iou':inter/max(union,1),'overreach_pixels':over,'missed_reviewed_pixels':miss,'overreach_fraction_of_candidate':over/max(int(c.sum()),1),'miss_fraction_of_reviewed':miss/max(int(r.sum()),1),'xor_pixels':int((r^c).sum())}
mask_rows=[]; mask_runtime={}
for _,r in cohort.iterrows():
    page=r.page; obs=cv2.imread(r.observed,0); rev=binary(cv2.imread(r.reviewed_mask,0)); raw=binary(cv2.imread(r.raw_yolo_mask,0)); conf=np.load(r.confidence_map).astype(np.float32)
    arms={'reviewed':rev,'raw_yolo':raw}
    for t in CONFIDENCE_GATES: arms[f'yolo_conf_{t:.3f}']=confidence_gated(raw,conf,t)
    if isinstance(r.reviewer_corrected_yolo,str) and r.reviewer_corrected_yolo and Path(r.reviewer_corrected_yolo).exists(): arms['reviewer_corrected_yolo']=binary(cv2.imread(r.reviewer_corrected_yolo,0))
    mask_runtime[page]={'observed':obs,'confidence':conf,'arms':arms}
    d=MASKS/page; d.mkdir(parents=True,exist_ok=False)
    for arm,m in arms.items():
        cv2.imwrite(str(d/f'{arm}.png'),m); q=mask_metrics(rev,m); q.update({'page':page,'arm':arm,'mask_sha256':sha256_file(d/f'{arm}.png')}); mask_rows.append(q)
mask_df=pd.DataFrame(mask_rows); mask_df.to_csv(TABLES/'mask_transfer_metrics.csv',index=False)
display(mask_df.groupby('arm')[['iou','overreach_pixels','missed_reviewed_pixels','xor_pixels']].mean())


## 6. Run every mask arm through frozen V8.8 and assert arm-specific immutability


In [ ]:
output_rows=[]; output_runtime={}
for page,z in mask_runtime.items():
    output_runtime[page]={}
    for arm,mask in z['arms'].items():
        w=OUTPUTS/page/arm; w.mkdir(parents=True,exist_ok=False)
        result=v88mod.run_v88(z['observed'].copy(),mask.copy(),w)
        required={'corrected','m3','m5_remove','audit'}
        if required-set(result): raise RuntimeError(f'{page} {arm}: V8.8 result missing {sorted(required-set(result))}')
        corrected=np.asarray(result['corrected']); m3=np.asarray(result['m3']); m5=np.asarray(result['m5_remove'])
        if corrected.shape!=z['observed'].shape or m3.shape!=corrected.shape or m5.shape!=corrected.shape: raise RuntimeError(f'{page} {arm}: V8.8 output shape mismatch')
        outside=int(np.sum(corrected[mask==0]!=z['observed'][mask==0]))
        if outside: raise RuntimeError(f'{page} {arm}: outside-mask drift={outside}')
        cv2.imwrite(str(w/'corrected.png'),corrected); cv2.imwrite(str(w/'m3.png'),m3); cv2.imwrite(str(w/'m5_remove.png'),m5)
        (w/'v88_audit.json').write_text(json.dumps(result['audit'],indent=2,default=str))
        output_runtime[page][arm]={'corrected':corrected,'m3':m3,'m5_remove':m5,'mask':mask}
        output_rows.append({'page':page,'arm':arm,'outside_mask_drift':outside,'corrected_array_sha256':array_hash(corrected),'m3_array_sha256':array_hash(m3),'m5_remove_array_sha256':array_hash(m5),'corrected_file_sha256':sha256_file(w/'corrected.png')})
output_df=pd.DataFrame(output_rows); output_df.to_csv(AUDIT/'v88_arm_output_integrity.csv',index=False)
print('Completed V8.8 arm runs:',len(output_df))


## 7. Prediction/output lock before truth scoring or paired reviewed-mask outcome comparison


In [ ]:
prediction_lock={'created_utc':datetime.now(timezone.utc).isoformat(),'input_lock_sha256':sha256_file(INPUTS/'input_lock.json'),'mask_metrics_sha256':sha256_file(TABLES/'mask_transfer_metrics.csv'),'v88_output_integrity_sha256':sha256_file(AUDIT/'v88_arm_output_integrity.csv'),'pages':len(pages),'arms':sorted(mask_df.arm.unique()),'output_hashes':output_df.sort_values(['page','arm'])[['page','arm','corrected_array_sha256','m3_array_sha256','m5_remove_array_sha256']].to_dict('records')}
(AUDIT/'prediction_lock.json').write_text(json.dumps(prediction_lock,indent=2))
print('Prediction/output lock:',object_hash(prediction_lock))


## 8. Paired transfer metrics against reviewed-mask V8.8 output

Only after the lock do we compare each proposal arm with the reviewed-mask comparator. These are transfer/effect metrics, not an assertion that the reviewed mask is perfect truth.


In [ ]:
paired=[]
for page,arms in output_runtime.items():
    ref=arms['reviewed']['corrected']; revmask=arms['reviewed']['mask']>0
    for arm,q in arms.items():
        if arm=='reviewed': continue
        cand=q['corrected']; m=q['mask']>0; diff=cand!=ref
        paired.append({'page':page,'arm':arm,'output_xor_pixels':int(diff.sum()),'output_xor_inside_union_mask':int((diff&(m|revmask)).sum()),'candidate_changed_pixels':int((cand!=mask_runtime[page]['observed']).sum()),'reviewed_changed_pixels':int((ref!=mask_runtime[page]['observed']).sum()),'masked_boundary_xor_pixels':int((m^revmask).sum()),'candidate_outside_mask_drift':int(np.sum(cand[~m]!=mask_runtime[page]['observed'][~m]))})
paired_df=pd.DataFrame(paired); paired_df.to_csv(TABLES/'paired_output_transfer_metrics.csv',index=False)
display(paired_df.groupby('arm')[['output_xor_pixels','masked_boundary_xor_pixels','candidate_changed_pixels']].mean())


## 9. Clean-truth scoring: structural harm and residual annotation

Clean pages enter here for the first time. The score is post-output only. Metrics are deliberately reported at page and arm level and should be interpreted with page/source-group uncertainty, not as independent pixels.


In [ ]:
def printed_ink(im): return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def tolerant_f1(ref,pred,region,tol=1):
    ref=ref&region; pred=pred&region
    if not ref.sum(): return np.nan,np.nan,np.nan
    k=np.ones((2*tol+1,2*tol+1),np.uint8); rd=cv2.dilate(ref.astype(np.uint8),k)>0; pdil=cv2.dilate(pred.astype(np.uint8),k)>0
    p=float(np.sum(pred&rd)/max(pred.sum(),1)); r=float(np.sum(ref&pdil)/ref.sum()); return p,r,2*p*r/max(p+r,1e-12)
truth_rows=[]
for _,row in cohort.iterrows():
    page=row.page
    if not isinstance(row.clean,str) or not row.clean or not Path(row.clean).exists(): continue
    clean=cv2.imread(row.clean,0); obs=mask_runtime[page]['observed']; ink=printed_ink(clean); obs_dark=obs<200
    for arm,q in output_runtime[page].items():
        out=q['corrected']; mask=q['mask']>0; outink=printed_ink(out)
        _,ink_recall,ink_f1=tolerant_f1(ink,outink,mask,1)
        removed_print=int(np.sum(mask & ink & (obs_dark) & (out>=200)))
        residual_dark=int(np.sum(mask & ~ink & (out<200)))
        truth_rows.append({'page':page,'arm':arm,'mask_pixels':int(mask.sum()),'ink_recall':ink_recall,'ink_f1':ink_f1,'removed_print_pixels_proxy':removed_print,'residual_dark_nonprint_pixels_proxy':residual_dark,'mae_inside_mask':float(np.abs(clean.astype(np.float32)-out.astype(np.float32))[mask].mean()) if mask.any() else np.nan})
truth_df=pd.DataFrame(truth_rows); truth_df.to_csv(TABLES/'posthoc_truth_scoring_by_page_arm.csv',index=False)
display(truth_df.groupby('arm')[['ink_recall','ink_f1','removed_print_pixels_proxy','residual_dark_nonprint_pixels_proxy','mae_inside_mask']].mean())


## 10. Reviewer edit-distance arm evidence

Where a reviewer-corrected YOLO mask exists, quantify how much human intervention was required to turn the raw proposal into the corrected YOLO mask and how much downstream V8.8 output changed.


In [ ]:
edit_rows=[]
for page,z in mask_runtime.items():
    if 'reviewer_corrected_yolo' not in z['arms']: continue
    raw=z['arms']['raw_yolo']>0; corr=z['arms']['reviewer_corrected_yolo']>0
    add=int((corr&~raw).sum()); erase=int((raw&~corr).sum()); xor=int((raw^corr).sum())
    out_raw=output_runtime[page]['raw_yolo']['corrected']; out_corr=output_runtime[page]['reviewer_corrected_yolo']['corrected']
    edit_rows.append({'page':page,'raw_pixels':int(raw.sum()),'corrected_pixels':int(corr.sum()),'reviewer_add_pixels':add,'reviewer_erase_pixels':erase,'mask_edit_xor_pixels':xor,'mask_edit_fraction_of_union':xor/max(int((raw|corr).sum()),1),'downstream_output_xor_pixels':int((out_raw!=out_corr).sum())})
edit_df=pd.DataFrame(edit_rows); edit_df.to_csv(TABLES/'reviewer_edit_distance.csv',index=False)
if len(edit_df): display(edit_df.describe(include='all'))
else: print('No reviewer-corrected YOLO mask arm available in this cohort.')


## 11. Page-clustered paired summaries and confidence-gate frontier


In [ ]:
merged=mask_df.merge(paired_df,on=['page','arm'],how='left').merge(truth_df,on=['page','arm'],how='left')
merged.to_csv(TABLES/'paired_mask_output_truth_evidence.csv',index=False)
summary_rows=[]
for arm,g in merged[merged.arm!='reviewed'].groupby('arm'):
    summary_rows.append({'arm':arm,'pages':g.page.nunique(),'mean_mask_iou':g.iou.mean(),'total_overreach_pixels':int(g.overreach_pixels.sum()),'total_missed_reviewed_pixels':int(g.missed_reviewed_pixels.sum()),'total_output_xor_pixels':int(g.output_xor_pixels.sum()),'mean_ink_recall':g.ink_recall.mean(),'mean_ink_f1':g.ink_f1.mean(),'total_removed_print_pixels_proxy':int(g.removed_print_pixels_proxy.fillna(0).sum()),'total_residual_dark_nonprint_pixels_proxy':int(g.residual_dark_nonprint_pixels_proxy.fillna(0).sum())})
summary_df=pd.DataFrame(summary_rows); summary_df.to_csv(TABLES/'arm_summary.csv',index=False)
display(summary_df)


## 12. Visual paired audit queues


In [ ]:
merged['review_priority']=merged.output_xor_pixels.fillna(0)+5*merged.overreach_pixels+5*merged.removed_print_pixels_proxy.fillna(0)
queue=merged[merged.arm!='reviewed'].sort_values('review_priority',ascending=False)
queue.to_csv(TABLES/'visual_review_queue.csv',index=False)

def contact(page,arm):
    obs=mask_runtime[page]['observed']; ref=output_runtime[page]['reviewed']['corrected']; cand=output_runtime[page][arm]['corrected']; rm=mask_runtime[page]['arms']['reviewed']>0; cm=mask_runtime[page]['arms'][arm]>0
    overlay=cv2.cvtColor(obs,cv2.COLOR_GRAY2RGB); overlay[rm&~cm]=(255,165,0); overlay[cm&~rm]=(255,0,0); overlay[rm&cm]=(0,180,0)
    fig,axs=plt.subplots(1,4,figsize=(20,7))
    for ax,im,title in zip(axs,[obs,overlay,ref,cand],['Observed','Mask agreement: green overlap / orange missed / red overreach','V8.8 with reviewed mask',f'V8.8 with {arm}']): ax.imshow(im,cmap='gray' if im.ndim==2 else None); ax.set_title(title); ax.axis('off')
    fig.suptitle(f'{page} | {arm}'); plt.tight_layout(); return fig
for i,r in queue.head(20).reset_index(drop=True).iterrows():
    fig=contact(r.page,r.arm); fig.savefig(VISUAL/f'{i+1:03d}_{r.page}_{r.arm}.png',dpi=130,bbox_inches='tight'); plt.close(fig)
print('Visual review queue:',VISUAL)


## 13. Fail-closed interpretation and V9.0A verdict

V9.0A does **not** invent an authority threshold from these development results. It reports whether each arm is technically transferable and whether any observed damage concentration blocks progression. A candidate automatic authority tier requires separately predeclared acceptance limits and fresh independent evidence.


In [ ]:
checks={
 'frozen_v88_identity_verified':True,
 'frozen_yolo_identity_verified':True,
 'paired_roster_complete':len(cohort)>0,
 'all_arm_outside_mask_drift_zero':bool((output_df.outside_mask_drift==0).all()),
 'prediction_lock_written':(AUDIT/'prediction_lock.json').exists(),
 'raw_yolo_is_shadow_only':True,
 'reviewed_mask_comparator_preserved':True,
}
raw_evidence=summary_df[summary_df.arm=='raw_yolo']
if len(raw_evidence):
    raw_evidence=raw_evidence.iloc[0]
    evidence_status='MEASURED_SHADOW_TRANSFER'
else: evidence_status='NO_RAW_YOLO_EVIDENCE'
verdict={'version':'9.0A','experiment':'YOLO Mask Authority and Transfer Audit','created_utc':datetime.now(timezone.utc).isoformat(),'checks':checks,'evidence_status':evidence_status,'automatic_mask_authority_authorised':False,'interpretation':'Shadow/development evidence only. Raw YOLO masks remain proposals unless a separately frozen authority policy passes independent validation.','v88_identity_sha256':sha256_file(v88_identity_file),'v88_module_sha256':V88_MODULE_SHA256,'yolo_identity_sha256':sha256_file(yolo_identity_file),'input_lock_sha256':sha256_file(INPUTS/'input_lock.json'),'prediction_lock_sha256':sha256_file(AUDIT/'prediction_lock.json')}
(OUT/'v9_0a_verdict.json').write_text(json.dumps(verdict,indent=2))
print(json.dumps(verdict,indent=2))


## 14. Final Upload Package

This curated package contains the result summary, paired evidence tables, audit locks, identities and the highest-priority visual review cases. Heavy per-arm working artefacts remain in the run directory and are referenced by hashes.


In [ ]:
import zipfile
summary_text="""V9.0A YOLO MASK AUTHORITY AND TRANSFER AUDIT

Development/shadow experiment only. V8.8 remained frozen. Raw YOLO masks were proposals, not automatic authority.

Review first:
- v9_0a_verdict.json
- tables/arm_summary.csv
- tables/paired_mask_output_truth_evidence.csv
- tables/reviewer_edit_distance.csv
- tables/visual_review_queue.csv
- audit/prediction_lock.json
- audit/v88_arm_output_integrity.csv
- visual_audit/

No result from this notebook by itself authorises automatic mask authority.
"""
(UPLOAD/'README.txt').write_text(summary_text)
curated=[OUT/'v9_0a_verdict.json',INPUTS/'input_lock.json',AUDIT/'prediction_lock.json',AUDIT/'v88_arm_output_integrity.csv',TABLES/'mask_transfer_metrics.csv',TABLES/'paired_output_transfer_metrics.csv',TABLES/'posthoc_truth_scoring_by_page_arm.csv',TABLES/'reviewer_edit_distance.csv',TABLES/'paired_mask_output_truth_evidence.csv',TABLES/'arm_summary.csv',TABLES/'visual_review_queue.csv']
for p in sorted(VISUAL.glob('*.png'))[:20]: curated.append(p)
manifest=[]
for src in curated:
    if not src.exists(): continue
    rel=('visuals/'+src.name) if src.suffix.lower()=='.png' else ('tables/'+src.name if src.parent==TABLES else 'audit/'+src.name if src.parent==AUDIT else src.name)
    dst=UPLOAD/rel; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst); manifest.append({'relative_path':str(dst.relative_to(UPLOAD)),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'upload_file_list.txt').write_text('
'.join(x['relative_path'] for x in manifest))
zip_path=UPLOAD/'v9_0a_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.rglob('*'):
        if p.is_file() and p!=zip_path: z.write(p,p.relative_to(UPLOAD))
if not zip_path.exists(): raise RuntimeError('Upload bundle creation failed.')
print('Upload bundle:',zip_path)


## Outputs to inspect first
- `v9_0a_verdict.json`
- `tables/arm_summary.csv`
- `tables/mask_transfer_metrics.csv`
- `tables/paired_output_transfer_metrics.csv`
- `tables/posthoc_truth_scoring_by_page_arm.csv`
- `tables/reviewer_edit_distance.csv`
- `tables/visual_review_queue.csv`
- `visual_audit/*.png`
- `upload/v9_0a_upload_bundle.zip`

**Interpretation:** a good raw-YOLO result is evidence for the next mask-authority policy experiment. It is not itself an authority rule. A poor result should drive detector/mask failure analysis, not relaxation of V8.8 correction safety.
